# Notebook 01 — Data Exploration

**Project:** Design and Implementation of a Deep Learning Intrusion Detection System Using LSTM  
**Chapter:** 3 — Materials and Methods (Section 3.5.1 — Data Acquisition and Exploratory Analysis)

## Objectives
- Load the raw NSL-KDD dataset
- Understand the dataset structure, feature types, and class distribution
- Identify missing values, duplicates, and data quality issues
- Generate all EDA figures required for Chapter 3 documentation

---

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.helpers import set_global_seed, print_banner
from src.utils.logger import get_logger
from src.config import get_config

cfg = get_config()
set_global_seed(cfg.seed)
logger = get_logger('notebook_01')

print_banner('Notebook 01 — Data Exploration')
print(f'Active dataset : {cfg.active_dataset}')
print(f'Random seed    : {cfg.seed}')

## 1. Dataset Availability Check

In [ ]:
from src.data.download import check_dataset_availability, acquire_dataset

availability = check_dataset_availability()
print('Dataset Availability:')
for ds, ok in availability.items():
    status = '✓ READY' if ok else '✗ NOT FOUND'
    print(f'  {ds:<20} {status}')

# Auto-download NSL-KDD if not present
if not availability.get('nsl_kdd', False):
    print('\nDownloading NSL-KDD...')
    acquire_dataset('nsl_kdd')

## 2. Load Raw NSL-KDD Dataset

In [ ]:
from src.data.loaders import load_nsl_kdd, get_nsl_kdd_summary

# Load separate train and test splits (no merge yet)
train_df, test_df = load_nsl_kdd(merge=False, validate=True)

print(f'Training set : {train_df.shape[0]:>7,} rows × {train_df.shape[1]} columns')
print(f'Test set     : {test_df.shape[0]:>7,} rows × {test_df.shape[1]} columns')

## 3. Dataset Structure

In [ ]:
print('Column dtypes:')
print(train_df.dtypes.value_counts())
print('\nFirst 3 rows:')
train_df.head(3)

In [ ]:
print('Training set statistics:')
train_df.describe().round(4)

## 4. Dataset Summary Statistics

In [ ]:
summary = get_nsl_kdd_summary(train_df)
print('NSL-KDD Dataset Summary:')
for k, v in summary.items():
    if not isinstance(v, dict):
        print(f'  {k:<35} {v}')

## 5. Raw Attack-Type Distribution

In [ ]:
from src.utils.constants import NSL_KDD_ATTACK_TO_CATEGORY, NSL_KDD_CLASS_NAMES

# Map raw labels to 5 categories
train_df['category'] = (
    train_df['label']
    .str.lower().str.strip()
    .map(NSL_KDD_ATTACK_TO_CATEGORY)
    .fillna('unknown')
)

cat_dist = train_df['category'].value_counts()
print('5-Class Category Distribution (Training):')
for cat, cnt in cat_dist.items():
    pct = cnt / len(train_df) * 100
    print(f'  {cat:<10} {cnt:>8,}  ({pct:5.2f}%)')

## 6. Class Distribution Plot

In [ ]:
from src.data.exploratory import plot_class_distribution

path = plot_class_distribution(train_df, dataset='nsl_kdd')
print(f'Figure saved: {path}')

from IPython.display import Image
Image(str(path))

## 7. Missing Value Analysis

In [ ]:
missing = train_df.isnull().sum()
total_missing = missing.sum()
print(f'Total missing values: {total_missing:,}')

if total_missing == 0:
    print('No missing values in NSL-KDD training set. ✓')
else:
    print(missing[missing > 0])

## 8. Duplicate Row Analysis

In [ ]:
n_dups = train_df.duplicated().sum()
print(f'Duplicate rows in training set: {n_dups:,} ({n_dups/len(train_df)*100:.2f}%)')

## 9. Categorical Feature Cardinalities

In [ ]:
from src.utils.constants import NSL_KDD_CATEGORICAL_FEATURES

print('Categorical feature cardinalities:')
for col in NSL_KDD_CATEGORICAL_FEATURES:
    n_unique = train_df[col].nunique()
    vals = train_df[col].value_counts().head(5).to_dict()
    print(f'  {col:<20} {n_unique} unique values | top-5: {vals}')

## 10. Correlation Heatmap

In [ ]:
from src.data.exploratory import plot_correlation_heatmap

path = plot_correlation_heatmap(train_df, dataset='nsl_kdd', top_n_features=20)
print(f'Figure saved: {path}')
Image(str(path))

## 11. Feature Distribution Histograms

In [ ]:
from src.data.exploratory import plot_feature_distributions

path = plot_feature_distributions(train_df, dataset='nsl_kdd', top_n=12)
print(f'Figure saved: {path}')
Image(str(path))

## 12. Validation Report

In [ ]:
from src.data.validators import validate_nsl_kdd_dataframe

report = validate_nsl_kdd_dataframe(train_df, split='train')
print(report.summary())

## 13. Full EDA Pipeline

In [ ]:
from src.data.exploratory import run_eda

saved_figures = run_eda(train_df, dataset='nsl_kdd')
print(f'\nAll EDA figures saved ({len(saved_figures)} total):')
for name, path in saved_figures.items():
    print(f'  {name:<30} → {path}')

---
**Summary:** The NSL-KDD dataset has been successfully loaded, validated, and explored.  
The class distribution plot, correlation heatmap, and feature histograms have been saved to `reports/figures/`  
and are ready for inclusion in **Chapter 3** of the project report.

**Next:** Run `02_data_preprocessing.ipynb`